# Exercise 2: Build a PyPSA Model

**Presenter** <br>
Priyesh Gosai <br>
Energy Systems Modeller <br>
priyesh@innovateimpact.com <br>

**Course Details**<br>
Modelling Integrated Power Markets <br>
Johannesburg, South Africa <br>
20-24 April 2026 <br>


## Prepare Google Colab Environment

In [ ]:
lesson_number = 2
print(f'lesson{lesson_number}')

In [ ]:
#@title Connect to Google Drive {display-mode:"form"}
CONNECT_TO_DRIVE = False #@param {type:"boolean"}

import os

if CONNECT_TO_DRIVE:
    from google.colab import drive
    # Mount Google Drive
    drive.mount('/content/drive')

    # Define the desired working directory path
    working_dir = f'/content/drive/MyDrive/ich-modelling-2026'
    lesson_folder = f'lesson-{lesson_number}'
    lesson_dir = os.path.join(working_dir, lesson_folder)

    # Create the working directory if it doesn't exist
    if not os.path.exists(working_dir):
        os.makedirs(working_dir)
        print(f"Directory '{working_dir}' created.")
    else:
        print(f"Directory '{working_dir}' already exists.")

    # Create the lesson directory if it doesn't exist
    if not os.path.exists(lesson_dir):
        os.makedirs(lesson_dir)
        print(f"Directory '{lesson_dir}' created.")
    else:
        print(f"Directory '{lesson_dir}' already exists.")

    # Change the current working directory to the lesson directory
    os.chdir(lesson_dir)

    print(f"Current working directory: {os.getcwd()}")
else:
    print("Not connecting to Google Drive.")

In [ ]:
#@title Install Packages {display-mode:"form"}
INSTALL_PACKAGES = False #@param {type:"boolean"}

import os

# Check if packages have already been installed in this session to prevent re-installation
if INSTALL_PACKAGES and not os.environ.get('PYPSA_PACKAGES_INSTALLED'):
  !pip install git+https://github.com/pypsa/pypsa
  !pip install pypsa[excel]
  !pip install folium mapclassify cartopy gurobipy
  os.environ['PYPSA_PACKAGES_INSTALLED'] = 'true'
elif not INSTALL_PACKAGES:
  print("Skipping package installation.")
else:
  print("PyPSA packages are already installed for this session.")

#### Case Description

This cases demonstrates the method for building a PyPSA network programatically.


##### PyPSA Features Used

- Creating a network
- Accessing the components using the API
- Viewing static data
- View dynamic data
- Run the optimiser
- Inspect results
- Plot results


##### Non-Standard PyPSA Features

This notebook will only use standard features in PyPSA.

### Lesson Task

Access the static and dynamic data in each cell.

* Identify the data types
* Access specific columns and rows in the data.
* Plot data.


1. Create the network object.

In [ ]:
import pypsa
import pandas as pd
pypsa.options.api.new_components_api = True

import numpy as np

n = pypsa.Network()

2. Set snapshots

In [ ]:
snapshots = pd.date_range('2025-01-01', '2025-12-31 23:00', freq='h')

n.set_snapshots(snapshots)

3. Add carriers

In [ ]:
carrier_types = ['wind','gas','AC']

n.add('Carrier',carrier_types)

4. Add a national bus

In [ ]:
n.add('Bus',
      'National Bus',
      carrier = 'AC',
      location = 'Region A')

5. Add a generators two loads

In [ ]:
n.add('Generator',
      'CCGT',
      bus = 'National Bus',
      p_nom = 500,
      marginal_cost = 60,
      )

n.add('Load',
      'Fixed Load',
      bus = 'National Bus',
      p_set = 100,
      carrier = 'AC')

n.add('Load',
      'Variable Load',
      bus = 'National Bus',
      carrier = 'AC')

variable_load = pd.DataFrame(np.random.randint(100, 251, size=len(n.snapshots)),
                             index=n.snapshots,
                             columns=['Variable Load'])

n.loads.dynamic.p_set.loc[:, 'Variable Load'] = variable_load['Variable Load']


6. Run the optimization to check if the model is working.

In [ ]:
n.optimize(include_objective_constant = True,
           solver_name = 'highs')

7. Add a battery as a storage unit.

In [ ]:
n.add('Bus',
      'Battery Bus',
      carrier = 'AC',
      location = 'Region A')

n.add('Store',
      'Battery',
      bus = 'Battery Bus',
      e_nom = 50)

n.add('Link',
      'Battery Link',
      bus0 = 'Battery Bus',
      bus1 = 'National Bus',
      p_nom = 50,
      p_min_pu = -1)


8. Run the optimization in two steps by creating the model, inspecting the model and solving the model.

In [ ]:
n.optimize.create_model()

In [ ]:
n.model

In [ ]:
n.optimize.solve_model()

Optimization is a two step process.

Therefore:

```

n.optimize()

```

is the same as:


```

n.create_model()

n.solve_model()

```








9. Inspect the results.

10. Export the solved model to an excel file.

In [ ]:
n.export_to_excel(f'lesson-{lesson_number}-solved.xlsx')